<img src="https://pyvoa.org/wp-content/uploads/2024/07/logo-pyvoa-1030x325.png" alt="PYVOA" width="125">
<center>

# The geo module of pyvoa, by examples
Tristan Beau

</center>

`pyvoa.geo` is what lets pyvoa understand `where='European Union'` or `where='Bretagne'`. Four classes:

- [`GeoCountry`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoCountry): the regions and subregions of one country, with their geometry;
- [`GeoInfo`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoInfo): facts about countries (population, area, capital…), added to a table;
- [`GeoManager`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoManager): turns any spelling of a country into a standard name or code;
- [`GeoRegion`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoRegion): the groups of countries (continents, European Union, G20…).

Their files are read from the pyvoa archive on Zenodo and kept on this machine after the first use.
Full reference: [pyvoa.geo](https://pyvoa.github.io/pyvoa/api/geo.html).

In [ ]:
# Run once on Colab / Binder, or on a fresh environment.
# %pip install pyvoa

In [ ]:
import geopandas as gpd
import pandas as pd

import pyvoa.geo as pg
from pyvoa.tools import PyvoaError

## GeoCountry: inside a country

`GeoCountry('FRA')` loads the geography of a country, given by its ISO 3166 alpha-3 code.
`get_list_countries()` lists those pyvoa knows ([doc](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoCountry)).

In [ ]:
pg.GeoCountry().get_list_countries()

In [ ]:
fra = pg.GeoCountry('FRA')
fra.get_country()

What the geography carries: its source files ([`get_source`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoCountry.get_source)), its
columns ([`get_list_properties`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoCountry.get_list_properties)), its regions and subregions —
for France, the régions and the départements.

In [ ]:
fra.get_source()

In [ ]:
fra.get_list_properties()

In [ ]:
fra.get_region_list()

In [ ]:
fra.get_subregion_list()

From a region to its subregions, by name — the case does not matter — or by code
([doc](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoCountry.get_subregions_from_region)).

In [ ]:
fra.get_subregions_from_region(name='Bretagne')

In [ ]:
fra.get_subregions_from_region(name='BretagnE')   # same answer

In [ ]:
fra.get_subregions_from_region(code='11')         # Île-de-France

In [ ]:
fra.get_subregions_from_list_of_region_names(['Normandie', 'Hauts-de-France'])

[`get_data()`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoCountry.get_data) returns the geography as a GeoDataFrame, by subregion;
`get_data(True)` by region. [`set_dense_geometry()`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoCountry.set_dense_geometry) brings the overseas
parts next to the mainland, as `map(typeofmap='dense')` does (France and the USA).

In [ ]:
fra.get_data()

In [ ]:
fra.get_data().plot()

In [ ]:
fra.set_dense_geometry()
fra.get_data().plot()

## GeoInfo: facts about countries

`get_list_field()` lists what can be added; `get_source()` says where each comes from
([doc](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoInfo)).

In [ ]:
gi = pg.GeoInfo()
gi.get_list_field()

In [ ]:
gi.get_source()

In [ ]:
gi.get_source('geometry')

[`add_field()`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoInfo.add_field) adds one field or a list of them to a table whose
`where` column names countries, in any spelling GeoManager understands. `overload=True` replaces a
column that exists already; `geofield` names the column of countries when it is not `where`.

In [ ]:
frame = pd.DataFrame({'where': ['France', 'Italy', 'Germany', 'Tunisia', 'Egypt'],
                      'value': [1, 2, 3, 4, 5]})
frame

In [ ]:
gi.add_field(input=frame, field='population')

In [ ]:
gi.add_field(input=frame, field=['capital', 'region_name_list', 'population', 'region_code_list'])

In [ ]:
frame2 = frame.assign(capital=['a', 'b', 'c', 'd', 'e'])
gi.add_field(input=frame2, field='capital', overload=True)

In [ ]:
frame3 = frame.rename(columns={'where': 'here'})
gi.add_field(input=frame3, field='area', geofield='here')

With the `geometry` field, a list of countries becomes a map:

In [ ]:
countries = pg.GeoRegion().get_countries_from_region('commonwealth')
gpd.GeoDataFrame(gi.add_field(input=pd.DataFrame({'where': countries}), field='geometry')).plot()

## GeoManager: one name for every spelling

A GeoManager converts country names to a *standard*: `'iso2'` (default), `'iso3'`, `'name'` or `'num'`
([`get_list_standard`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoManager.get_list_standard)), given when it is created or with
[`set_standard`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoManager.set_standard)).

In [ ]:
gm = pg.GeoManager()
gm.get_list_standard()

In [ ]:
gm.set_standard('name')
gm.get_standard()

In [ ]:
pg.GeoManager('iso3').get_standard()

[`to_standard()`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoManager.to_standard) takes a name or a list of them, in English or in ISO
codes. A name that is not a country raises a `PyvoaError`, unless `interpret_region=True` lets it stand
for the countries of a region. An ambiguous name gives its first match, with a warning.

In [ ]:
gm.to_standard('england')

In [ ]:
gm.to_standard(['esp', 'it', 'Berlin'])

In [ ]:
try:
    gm.to_standard('European Union')
except PyvoaError:
    print('not a country')
gm.to_standard('European Union', interpret_region=True)

In [ ]:
gm.to_standard(['usa', 'South America', 'russia'], interpret_region=True)

In [ ]:
gm.to_standard('french')   # ambiguous: first match, with a warning

Some databases spell countries their own way; `db=` reads names as that database writes them
([`get_list_db`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoManager.get_list_db)). `output` chooses what comes back: `'list'` (default),
`'dict'` or `'pandas'` ([`get_list_output`](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoManager.get_list_output)).

In [ ]:
gm.get_list_db()

In [ ]:
gm.to_standard('Congo (Kinshasa)', db='jhu')

In [ ]:
gm.to_standard(['fr', 'spain', 'england'], output='dict')

In [ ]:
gm.to_standard(['fr', 'spain', 'england'], output='pandas')

## GeoRegion: groups of countries

Continents and sub-continents (UN M49 and UN geoscheme), economic and political unions
([doc](https://pyvoa.github.io/pyvoa/api/geo.html#pyvoa.geo.GeoRegion)). These are the names `where=` accepts for a region.

In [ ]:
gr = pg.GeoRegion()
gr.get_region_list()

In [ ]:
gr.get_source()

In [ ]:
gr.get_countries_from_region('South America')

In [ ]:
gr.get_pandas()